# TMol How-To Guide — Selections

## 5. Selections

Every selection in TMol is a plain boolean `torch.Tensor` -- block-shaped (`[n_poses, max_n_blocks]`) or atom-shaped (`[n_poses, max_n_atoms]`) -- built by hand from whatever a `PoseStack` already exposes. There's no dedicated "Selection" class or query language, just tensors and boolean algebra; this notebook collects the common patterns in one place.

### Setup

Wheel install, imports, seeding, and device selection live in [notebook_setup.py](notebook_setup.py).

In [1]:
try:
    import google.colab  # noqa: F401
except ImportError:
    IN_COLAB = False
else:
    IN_COLAB = True

if IN_COLAB:
    from urllib.request import urlretrieve

    urlretrieve(
        "https://raw.githubusercontent.com/uw-ipd/tmol/"
        "master/notebooks/notebook_setup.py",
        "notebook_setup.py",
    )

from notebook_setup import *  # noqa: F401,F403

device, param_db = setup()


device=cpu


`pose_stack.pdb_info` is where most chain/residue-identity selections start: parallel `[pose, block]` arrays `chain_labels`, `residue_labels`, and `residue_insertion_codes`, plus per-atom `atom_occupancy` and `atom_b_factor`. We use [1HDD](https://www.rcsb.org/structure/1HDD) (a homeodomain-DNA complex) from section 3 as the running example, for its multiple chains and two polymer types.

In [2]:
cif_text = fetch_cif("1HDD")
pose_stack = pose_stack_from_cif(io.StringIO(cif_text), device, param_db=param_db)
print(f"n_poses={pose_stack.n_poses}, max_n_blocks={pose_stack.max_n_blocks}")


ARG is seen only at a chain terminus and N, NE, NH1, NH2 could each carry its other connection; it is read as the conventional backbone through N. A copy of the residue in a chain would settle it.


n_poses=1, max_n_blocks=156


### Select by chain

There is no chain-selection helper; compare `pdb_info.chain_labels` directly. (`pose_stack.chain_id64` gives an integer per-pose chain index, if comparing against a label string is inconvenient.)

In [3]:
chain_labels = pose_stack.pdb_info.chain_labels[0]
chain_a = torch.tensor([label == "A" for label in chain_labels], device=device).unsqueeze(0)
print(f"unique chains: {sorted(set(chain_labels.tolist()))}")
print(f"chain A blocks: {int(chain_a.sum())}")


unique chains: ['A', 'B', 'C', 'D']
chain A blocks: 21


### Select by backbone / side chain

`build_sidechain_coord_mask` is atom-shaped (`[n_poses, max_n_atoms]`) and excludes each residue type's declared main-chain atoms (non-polymers count as all side chain). There's no separate backbone function -- it's the complement, restricted to real atoms.

In [4]:
sidechain_atoms = build_sidechain_coord_mask(pose_stack)
backbone_atoms = pose_stack.real_atoms & ~sidechain_atoms
print(f"sidechain atoms: {int(sidechain_atoms.sum())}")
print(f"backbone atoms:  {int(backbone_atoms.sum())}")


sidechain atoms: 2748
backbone atoms:  592


### Select by polymer type

Each block type declares `properties.polymer.polymer_type` (`"amino_acid"`, `"nucleic_acid"`, or `None` for a non-polymer like a ligand or ion), alongside the `is_polymer` flag already used in section 1. There's no vectorized version -- walk the blocks and look each one up.

In [5]:
def block_mask_by_polymer_type(pose_stack, polymer_type):
    mask = torch.zeros((pose_stack.n_poses, pose_stack.max_n_blocks), dtype=torch.bool)
    for pose in range(pose_stack.n_poses):
        for block in range(pose_stack.max_n_blocks):
            bt_ind = int(pose_stack.block_type_ind64[pose, block])
            if bt_ind < 0:
                continue
            block_type = pose_stack.packed_block_types.active_block_types[bt_ind]
            mask[pose, block] = block_type.properties.polymer.polymer_type == polymer_type
    return mask


protein_blocks = block_mask_by_polymer_type(pose_stack, "amino_acid")
dna_blocks = block_mask_by_polymer_type(pose_stack, "nucleic_acid")
print(f"protein blocks: {int(protein_blocks.sum())}, DNA blocks: {int(dna_blocks.sum())}")


protein blocks: 114, DNA blocks: 42


### Select within *k* Å of a reference

There is no "within distance" utility either -- the real pattern throughout TMol's tutorials is a plain `torch.cdist` between all atom coordinates and the reference atoms' coordinates, thresholded. We write it once and reuse it for a DNA chain (already loaded), a ligand, a metal ion, and a non-canonical residue.

In [6]:
def atoms_within(pose_stack, reference_atom_mask, distance_angstroms, pose=0):
    """Real atoms within `distance_angstroms` of any atom in `reference_atom_mask`."""
    all_coords = torch.nan_to_num(pose_stack.coords[pose])
    reference_coords = pose_stack.coords[pose][reference_atom_mask[pose]]
    nearest_distance = torch.cdist(all_coords, reference_coords).min(dim=1).values
    return (nearest_distance <= distance_angstroms) & pose_stack.real_atoms[pose]


dna_coord_mask = res_mask_to_coord_mask(pose_stack, dna_blocks)
near_dna = atoms_within(pose_stack, dna_coord_mask, 5.0)
print(f"atoms within 5 A of DNA: {int(near_dna.sum())}")


atoms within 5 A of DNA: 1709


In [7]:
from tmol.io import atom_array_from_cif, pose_stack_from_biotite
from tmol.tests.data import data_path

# A ligand: benzamidine (BEN) bound to trypsin, from section 1/4.
ligand_pose_stack = pose_stack_from_biotite(
    atom_array_from_cif(io.StringIO(fetch_cif("3PTB"))), device, prepare_ligands=True, param_db=param_db
)
ligand_blocks = torch.zeros((1, ligand_pose_stack.max_n_blocks), dtype=torch.bool)
for block in range(ligand_pose_stack.max_n_blocks):
    if ligand_pose_stack.block_type(0, block).name3 == "BEN":
        ligand_blocks[0, block] = True
near_ligand = atoms_within(ligand_pose_stack, res_mask_to_coord_mask(ligand_pose_stack, ligand_blocks), 5.0)
print(f"atoms within 5 A of the ligand: {int(near_ligand.sum())}")

# A metal ion: the zinc site from section 3.
metal_pose_stack = pose_stack_from_biotite(
    atom_array_from_cif(data_path("metal_fixtures") / "zn_tetrahedral_3ks3.cif.zst"),
    device, prepare_ligands=True, param_db=param_db,
)
metal_blocks = torch.zeros((1, metal_pose_stack.max_n_blocks), dtype=torch.bool)
for block in range(metal_pose_stack.max_n_blocks):
    if metal_pose_stack.block_type(0, block).name3 == "ZN":
        metal_blocks[0, block] = True
near_metal = atoms_within(metal_pose_stack, res_mask_to_coord_mask(metal_pose_stack, metal_blocks), 5.0)
print(f"atoms within 5 A of the metal: {int(near_metal.sum())}")

# A non-canonical residue: AIB from section 4.
aib_pose_stack = pose_stack_from_biotite(
    atom_array_from_cif(data_path("ncaa_fixtures") / "uic1_aib_9bzt.cif"), device, param_db=param_db, no_optH=True
)
aib_blocks = torch.zeros((1, aib_pose_stack.max_n_blocks), dtype=torch.bool)
for block in range(aib_pose_stack.max_n_blocks):
    if aib_pose_stack.block_type(0, block).name3 == "AIB":
        aib_blocks[0, block] = True
near_aib = atoms_within(aib_pose_stack, res_mask_to_coord_mask(aib_pose_stack, aib_blocks), 5.0)
print(f"atoms within 5 A of any AIB residue: {int(near_aib.sum())}")


*** Open Babel Warning  in WriteMolecule
  No 2D or 3D coordinates exist. Stereochemical information will be stored using an Open Babel extension. To generate 2D or 3D coordinates instead use --gen2D or --gen3D.


atoms within 5 A of the ligand: 142


HIS is seen only at a chain terminus and N, ND1, NE2 could each carry its other connection; it is read as the conventional backbone through N. A copy of the residue in a chain would settle it.


atoms within 5 A of the metal: 103


atoms within 5 A of any AIB residue: 138


### Multiple criteria

Combining selections is plain boolean algebra on the tensors -- `|` for "or", `&` for "and", `~` for "not" -- no special combinator needed. Our 1HDD example has no metal site, so in its place we combine two distance criteria it does have: within 8 Å of the DNA, or within 5 Å of the second protein chain.

In [8]:
chain_b = torch.tensor([label == "B" for label in chain_labels], device=device).unsqueeze(0)

near_dna_8A = atoms_within(pose_stack, dna_coord_mask, 8.0)
near_chain_b_5A = atoms_within(pose_stack, res_mask_to_coord_mask(pose_stack, chain_b), 5.0)
combined = near_dna_8A | near_chain_b_5A

print(f"within 8 A of DNA: {int(near_dna_8A.sum())}")
print(f"within 5 A of chain B: {int(near_chain_b_5A.sum())}")
print(f"either condition: {int(combined.sum())}")


within 8 A of DNA: 2208
within 5 A of chain B: 1142
either condition: 2208
